In [1]:
# %pip install --user opencv-contrib-python-headless==4.11.0.86
# %pip install --user opencv-contrib-python==4.11.0.86 --force-reinstall


In [1]:
import cv2

print(cv2.__version__)
 

4.11.0


In [14]:
 
#Open Webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not access webcam.")
    exit()


In [15]:
#read first frame
ret, frame = cap.read()
if not ret:
    print("Failed to capture first frame.")
    cap.release()
    exit()


In [16]:
bbox = cv2.selectROI("Select Object", frame, False)
cv2.destroyWindow("Select Object")

print("Selected ROI:", bbox)

tracker = cv2.TrackerCSRT_create()
tracker.init(frame, bbox)



Selected ROI: (220, 293, 162, 146)


In [17]:
# Setup Video Recording
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps_record = 20
height, width = frame.shape[:2]

out = cv2.VideoWriter(
    "tracking_demo1.mp4",
    fourcc,
    fps_record,
    (width, height)
)

In [18]:
#FPS measurement setup
import time

start_time = time.time()
frame_count = 0

print("Tracking and recording started...")
print("Press ESC to stop.")


Tracking and recording started...
Press ESC to stop.


In [19]:
# real-time tracking
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Update tracker
    success, bbox = tracker.update(frame)

    # Draw bounding box if tracking succeeds
    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        status_text = "Tracking"
        status_color = (0, 255, 0)
    else:
        status_text = "Lost"
        status_color = (0, 0, 255)

    cv2.putText(frame, status_text, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1,
                status_color, 2)

    # Calculate FPS
    elapsed_time = time.time() - start_time
    fps = frame_count / elapsed_time if elapsed_time > 0 else 0

    cv2.putText(frame, f"FPS: {int(fps)}",
                (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 255, 0),
                2)

    # Show frame
    cv2.imshow("Real-Time Object Tracker", frame)

    # Save frame
    out.write(frame)

    # Exit on ESC
    if cv2.waitKey(1) & 0xFF == 27:
        break

In [20]:
#release everything 
cap.release()
out.release()
cv2.destroyAllWindows()

print("Recording saved as: tracking_demo1.mp4")

Recording saved as: tracking_demo1.mp4
